In [1]:
import os.path
import gc
import pandas as pd
import pickle as pkl
import numpy as np
import random
from tqdm import tqdm
import json
from sklearn import metrics
from utils import get_texts_for_ptype
from sklearn import metrics
from scipy.optimize import minimize_scalar
from custom_models.custom_model import BatchVectorizerClassifier
from ray import tune

In [ ]:
model_types = [
    "hinge", 
    "log_loss", 
    "modified_huber", 
    "squared_hinge", 
    "perceptron"]
neg_to_pos_ratio = 1

results_file = "results_titles_abstracts_linear_features1000.json"
linear_metrics_adjusted_results_path = "results_adjusted_metrics_linear_classifiers.json"

inputs = "titles_abstracts"

filename_pos_human = "../data/pubmed_articles_pos_human.csv"
filename_neg_human = "../data/pubmed_articles_neg_human.csv"

with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
    prop_typ_id_dict = pkl.load(f)

ptypes = prop_typ_id_dict.keys()

In [3]:
with open(results_file, "r") as f_read:
    results_list = json.load(f_read)

In [4]:
# decision_fn_file = "results_decision_fn_linear_features1000.json"

In [5]:
def predict(df, db=0):
    return [int(x>=db) for x in df ]

In [6]:
neg_pos_ratio_dict = {
    "is_rct": 5585042/489295,
    "is_sr": 5659292/121063,
    "is_casecontrol": 5628219/65790,
    "is_cohort": 5632460/78688,
    "is_guideline": 5676411/33826
}

def calc_precision(recall, specificity, neg_pos_ratio):
    return recall / (recall + neg_pos_ratio * (1 - specificity) + 1e-15)

def calc_f1(recall, precision):
    return 2*recall*precision/(recall+precision + 1e-15)

In [7]:
def find_db(db):
    y_pred_test = predict(df, db)
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    return abs(recall_test - 0.985)

In [8]:
metrics_lst = []
for results_json in tqdm(results_list):
    ptype = results_json["ptype"]
    model_type = results_json["model_type"]
    X_train, X_val, X_test, y_train, y_val, y_test = get_texts_for_ptype(ptype, inputs="titles_abstracts", neg_to_pos_ratio=neg_to_pos_ratio)    
    bvc = BatchVectorizerClassifier(
                    loss=results_json["model_type"],
                    penalty=results_json["best_penalty"],
                    alpha=results_json["best_alpha"],
                    l1_ratio=0.15,
                    fit_intercept=True,
                    max_iter=1000,
                    tol=0.001,
                    shuffle=True,
                    verbose=0,
                    epsilon=0.1,
                    n_jobs=None,
                    random_state=None,
                    learning_rate="optimal",
                    eta0=0.0,
                    power_t=0.5,
                    early_stopping=False,
                    validation_fraction=0.1,
                    n_iter_no_change=5,
                    class_weight=None,
                    warm_start=False,
                    average=False,
        )
    bvc.fit(X_train, y_train)
    df = bvc.decision_function(X_test)
    
    auc_roc = metrics.roc_auc_score(y_test, df)
    
    res = minimize_scalar(find_db)
    db = res["x"]
    y_pred_test = predict(df, db)
    
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
    precision_test = calc_precision(recall_test, spec_test, neg_pos_ratio_dict[ptype])
    f1_test = calc_f1(recall_test, precision_test)
    
    metrics_dict = {
        "ptype": ptype,
        "model_type": model_type,
        "y_test": y_test,
        "db": db,
        "df": df,
        "auc_roc": auc_roc,
        "recall_test": recall_test,
        "spec_test": spec_test,
        "precision_test": precision_test,
        "f1_test": f1_test
    }
    metrics_lst.append(metrics_dict)
    
    if os.path.exists(linear_metrics_adjusted_results_path):
        with open(linear_metrics_adjusted_results_path, "rb") as f:
            metrics_lst = pkl.load(f)
    
    metrics_lst.append(metrics_dict)
    with open(linear_metrics_adjusted_results_path, "wb") as f:
        pkl.dump(metrics_lst, f)
        
    del bvc
    gc.collect()

  0%|          | 0/25 [00:00<?, ?it/s]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [01:47,  1.25s/it]
5it [00:08,  1.69s/it]
  0%|          | 0/25 [01:56<?, ?it/s]


KeyboardInterrupt: 

In [12]:
metrics_lst

[{'ptype': 'is_rct',
  'model_type': 'hinge',
  'y_test': 0       0
  1       1
  2       0
  3       0
  4       1
         ..
  7995    0
  7996    1
  7997    0
  7998    1
  7999    1
  Name: is_rct, Length: 8000, dtype: int64,
  'db': -1.584230192290271,
  'df': array([-1.04781593,  3.99362064, -0.12792949, ..., -0.56673838,
          1.47217621,  1.63359905]),
  'auc_roc': 0.91500375,
  'recall_test': 0.985,
  'spec_test': 0.3135,
  'precision_test': 0.11166493754022432,
  'f1_test': 0.20058991531875534},
 {'ptype': 'is_rct',
  'model_type': 'log_loss',
  'y_test': 0       1
  1       0
  2       0
  3       0
  4       1
         ..
  7995    1
  7996    1
  7997    1
  7998    1
  7999    0
  Name: is_rct, Length: 8000, dtype: int64,
  'db': -1.7706254025899768,
  'df': array([ 2.56437989, -2.36902622, -0.9905455 , ...,  2.08079802,
         -0.29060715, -1.12184053]),
  'auc_roc': 0.9198798750000001,
  'recall_test': 0.985,
  'spec_test': 0.36725,
  'precision_test': 0.1200121